# BayesTextMMRL ECE 诊断 Notebook v6：确定性严格 / Bayes 记录差异版

这个版本修正前面 notebook 的关键问题：

1. **先跑 `trainer.test(split="test")` 复现官方 structured result**；
2. **再手工收集 logits / MC stack**；
3. **强制比较 official evaluator 指标与手工 `repo_logits` 指标**；
4. 对 FS 协议，强制检查：

```text
repo_logits == logits_fusion
```

因为在 FS 协议下，项目代码的 `select_eval_logits()` 应该返回 fusion logits。  
如果这个 sanity check 失败，notebook 会直接停止，避免继续输出误导性诊断表。

默认分析路径：

```text
/root/autodl-tmp/MMRL/output_refactor/MMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1
/root/autodl-tmp/MMRL/output_refactor/BayesTextMMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1
```

默认输出目录：

```text
/root/autodl-tmp/MMRL/output_refactor/analysis/bayes_text_mmrl_ece_diagnosis_v4_strict_sanity_dtd16_seed1
```

## 重要说明

这个 notebook 不再猜测加载的是 best 还是 last。  
它会读取真实日志文本，如果日志中包含：

```text
Deploy the last-epoch model
```

则加载最后一轮 checkpoint；如果没有该提示，则使用日志中的 `load_epoch` 或项目默认 `epoch=None` 行为。

> **v5 修正**
>
> v4 的 BayesTextMMRL sanity 失败，不一定是模型/日志不一致，而是 notebook 实现错误：
> 在同一个 batch 循环里先调用 official `forward_eval`，又立刻调用 posterior mean / diagnostic MC forward。
> 这些额外 forward 会消耗随机数，导致后续 batch 的 official MC samples 与 `trainer.test()` 的随机流不同。
>
> v5 改为两遍收集：
>
> 1. **official pass**：只调用 `method.forward_eval()`，严格复现 `trainer.test()`；
> 2. **diagnostic pass**：另起一遍，收集 posterior mean、MC stack、arithmetic fusion 等诊断量。
>
> 因此 sanity check 只比较 official pass 与 `trainer.test()`，不受 diagnostic forward 影响。

> **v6 修正：BayesTextMMRL 不再做 MC 逐次严格相等断言**
>
> `MMRL` 是确定性 eval，必须严格复现日志 / official evaluator。
>
> `BayesTextMMRL` 的 `forward_eval()` 会在推理时采样文本分类器：
>
> ```python
> out = self.model.forward_bayes_text(
>     image,
>     num_samples=self.n_mc_test,
>     use_posterior_mean=self.eval_use_posterior_mean,
>     aggregation=self._aggregation(),
> )
> ```
>
> 因此两次独立 eval pass 即使都是“正常脚本”，也可能因为 MC sampling 得到略不同 logits / ECE。
> 其他 repo notebook 通常只做一次 repo-forward 收集并分析，不要求它和另一次 `trainer.test()` 完全一致。
>
> 本版规则：
>
> - `MMRL`：official 与 manual 必须严格一致；
> - `BayesTextMMRL`：记录 official-vs-manual 差异，但不作为 hard error；
> - FS 协议下，仍检查同一次 forward 内 `repo_logits == fusion_logits`，这个应当成立。

In [ ]:
from pathlib import Path
import os
import sys
import re
import ast
import json
import math
import random
import importlib
from types import SimpleNamespace
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

# ===== 用户主要配置 =====
REPO_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()
DATA_ROOT = REPO_ROOT / "DATASETS"

MMRL_CASE_ROOT = REPO_ROOT / "output_refactor/MMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1"
BAYES_CASE_ROOT = REPO_ROOT / "output_refactor/BayesTextMMRL/FS/fewshot_train/dtd/shots_16/ViT-B-16/default/seed1"

OUT_DIR = REPO_ROOT / "output_refactor/analysis/bayes_text_mmrl_ece_diagnosis_v6_mmrldeterministic_bayesrecord_dtd16_seed1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["val", "test"]

# None 表示使用 checkpoint/config 中的 N_MC_TEST；若要强制更多采样，可设为 20/30/50。
N_MC_DIAG = None

# 调试时可设小整数，例如 5；正式分析设 None。
MAX_BATCHES = None

# 固定评估随机种子，降低 MC 重复运行波动。
EVAL_SEED = 2026

# ECE bins 与项目 evaluation/metrics.py 默认保持一致。
N_BINS = 10

# sanity check 是否严格中止。建议 True。
STRICT_OFFICIAL_SANITY = True

# official metrics 与手工 repo_logits 指标允许误差。
# MMRL 应接近 0；BayesTextMMRL 有 MC stochastic，但本 notebook 会重置 seed，通常也应很小。
SANITY_TOL_ACC = 1e-3
SANITY_TOL_ECE = 1e-3
SANITY_TOL_NLL = 1e-4

SAVE_NPZ_CACHE = True

print("REPO_ROOT =", REPO_ROOT)
print("MMRL_CASE_ROOT =", MMRL_CASE_ROOT)
print("BAYES_CASE_ROOT =", BAYES_CASE_ROOT)
print("OUT_DIR =", OUT_DIR)

assert REPO_ROOT.exists(), f"REPO_ROOT 不存在: {REPO_ROOT}"
assert MMRL_CASE_ROOT.exists(), f"MMRL_CASE_ROOT 不存在: {MMRL_CASE_ROOT}"
assert BAYES_CASE_ROOT.exists(), f"BAYES_CASE_ROOT 不存在: {BAYES_CASE_ROOT}"

## 1. 导入 repo 代码并注册方法

In [ ]:
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_THREADS", "1")
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from core.config import setup_cfg
from core.utils import import_optional_modules
from core.types import MethodOutputs
from dassl.engine import build_trainer
from dassl.utils import set_random_seed

import_optional_modules([
    "datasets.oxford_pets", "datasets.oxford_flowers", "datasets.fgvc_aircraft",
    "datasets.dtd", "datasets.eurosat", "datasets.stanford_cars", "datasets.food101",
    "datasets.sun397", "datasets.caltech101", "datasets.ucf101", "datasets.imagenet",
    "datasets.imagenetv2", "datasets.imagenet_sketch", "datasets.imagenet_a", "datasets.imagenet_r",
])
importlib.import_module("trainers.refactor_runner")

print("torch =", torch.__version__)
print("cuda available =", torch.cuda.is_available())

## 2. 真实日志解析与 checkpoint 选择

In [ ]:
def find_real_log_file(case_root: Path, explicit_log_file=None) -> Path:
    case_root = Path(case_root).expanduser().resolve()
    if explicit_log_file:
        p = Path(explicit_log_file).expanduser().resolve()
        if p.exists() and p.is_file():
            return p
        raise FileNotFoundError(f"显式指定 log_file 不存在: {p}")

    candidates = [
        case_root / "run.log",
        case_root / "log.txt",
    ]
    candidates += sorted(case_root.glob("log.txt-*"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)
    candidates += sorted(case_root.glob("*.log"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

    seen = set()
    unique = []
    for p in candidates:
        if str(p) not in seen:
            unique.append(p)
            seen.add(str(p))

    for p in unique:
        if not (p.exists() and p.is_file()):
            continue
        head = p.read_text(encoding="utf-8", errors="ignore")[:80000]
        if "** Arguments **" in head and "opts:" in head:
            return p

    raise FileNotFoundError(f"找不到包含 Arguments/opts 的训练日志: {case_root}")


def _parse_scalar_from_log_value(value: str):
    value = value.strip()
    if value == "None":
        return None
    if value == "":
        return ""
    if value in {"True", "False"}:
        return value == "True"
    if value.startswith("[") and value.endswith("]"):
        return ast.literal_eval(value)
    return value


def parse_args_from_log(log_path: Path) -> dict:
    text = Path(log_path).read_text(encoding="utf-8", errors="ignore")

    m = re.search(
        r"\*\* Arguments \*\*\s*\n\*+\s*\n(?P<body>.*?)(?:\n\*{4,}\s*\n\*\* Config \*\*|\nCollecting env info|\nLoading trainer:)",
        text,
        flags=re.S,
    )
    if m is None:
        raise ValueError(f"日志中找不到 Arguments 区块: {log_path}")

    parsed = {}
    for raw_line in m.group("body").splitlines():
        line = raw_line.rstrip()
        if not line or ": " not in line:
            continue
        key, value = line.split(": ", 1)
        parsed[key.strip()] = _parse_scalar_from_log_value(value.strip())

    if "opts" in parsed and isinstance(parsed["opts"], str):
        parsed["opts"] = ast.literal_eval(parsed["opts"])

    for key in ["eval_only", "no_train"]:
        if key in parsed and isinstance(parsed[key], str):
            parsed[key] = parsed[key] == "True"

    for key in ["seed", "load_epoch"]:
        if key in parsed:
            if parsed[key] in {None, ""}:
                parsed[key] = None
            else:
                parsed[key] = int(parsed[key])

    required = [
        "dataset_config_file",
        "method_config_file",
        "protocol_config_file",
        "runtime_config_file",
        "method",
        "protocol",
        "exec_mode",
        "seed",
        "trainer",
        "opts",
    ]
    missing = [k for k in required if k not in parsed]
    if missing:
        raise ValueError(f"日志 Arguments 缺少字段 {missing}: {log_path}")

    if not isinstance(parsed["opts"], list):
        raise TypeError(f"日志 opts 不是 list: {type(parsed['opts'])}, log={log_path}")

    return parsed


def find_last_checkpoint_epoch(case_root: Path):
    model_dir = Path(case_root).expanduser().resolve() / "refactor_model"
    if not model_dir.exists():
        raise FileNotFoundError(f"找不到 refactor_model 目录: {model_dir}")

    epochs = []
    for p in model_dir.glob("model.pth.tar-*"):
        m = re.search(r"model\.pth\.tar-(\d+)$", p.name)
        if m:
            epochs.append(int(m.group(1)))

    if not epochs:
        raise FileNotFoundError(f"找不到 model.pth.tar-<epoch> checkpoint: {model_dir}")

    return max(epochs)


def detect_checkpoint_policy_from_log(log_path: Path):
    text = Path(log_path).read_text(encoding="utf-8", errors="ignore")
    if "Deploy the last-epoch model" in text:
        return "last", "detected_log_text: Deploy the last-epoch model"
    if "Deploy the best model" in text or "Deploy the best-epoch model" in text:
        return "best", "detected_log_text: Deploy the best model"
    return "log_or_default", "no explicit deploy policy detected"


def resolve_load_epoch(case_root: Path, log_path: Path, parsed_args: dict):
    policy, reason = detect_checkpoint_policy_from_log(log_path)

    if policy == "last":
        ep = find_last_checkpoint_epoch(case_root)
        return ep, f"last_epoch_{ep}; {reason}"

    if policy == "best":
        return None, f"best_checkpoint; {reason}"

    # 没有明确 deploy 提示时，复用日志中的 load_epoch；若为 None，则交给项目默认 load_model 行为。
    ep = parsed_args.get("load_epoch", None)
    return ep, f"log_load_epoch_{ep}; {reason}"


def make_args_from_case(case_root: Path, explicit_log_file=None, output_dir_override=None):
    case_root = Path(case_root).expanduser().resolve()
    log_path = find_real_log_file(case_root, explicit_log_file)
    parsed = parse_args_from_log(log_path)
    load_epoch, load_epoch_source = resolve_load_epoch(case_root, log_path, parsed)

    args = SimpleNamespace()
    args.root = str(DATA_ROOT)
    args.output_dir = str(output_dir_override or case_root)
    args.dataset_config_file = str(parsed.get("dataset_config_file") or "")
    args.method_config_file = str(parsed.get("method_config_file") or "")
    args.protocol_config_file = str(parsed.get("protocol_config_file") or "")
    args.runtime_config_file = str(parsed.get("runtime_config_file") or "")
    args.exp_config = str(parsed.get("exp_config") or "")
    args.method = str(parsed.get("method") or "")
    args.protocol = str(parsed.get("protocol") or "")
    args.exec_mode = str(parsed.get("exec_mode") or "")
    args.seed = int(parsed["seed"]) if parsed.get("seed") is not None else -1
    args.trainer = str(parsed.get("trainer") or "RefactorRunner")
    args.eval_only = True
    args.no_train = True
    args.model_dir = str(case_root)
    args.load_epoch = load_epoch
    args.load_epoch_source = load_epoch_source
    args.opts = list(parsed.get("opts") or [])
    return args, log_path, parsed


def build_and_load_trainer(case_name: str, case_root: Path):
    args, log_path, parsed_args = make_args_from_case(
        case_root=case_root,
        output_dir_override=OUT_DIR / "official_eval_outputs" / case_name,
    )

    print(f"\n===== Build case: {case_name} =====")
    print("log_path =", log_path)
    print("method =", args.method, "protocol =", args.protocol, "exec_mode =", args.exec_mode)
    print("seed =", args.seed)
    print("model_dir =", args.model_dir)
    print("load_epoch =", args.load_epoch)
    print("load_epoch_source =", args.load_epoch_source)

    cfg = setup_cfg(args)
    if cfg.SEED >= 0:
        set_random_seed(cfg.SEED)

    trainer = build_trainer(cfg)
    trainer.load_model(args.model_dir, epoch=args.load_epoch)

    return trainer, args, parsed_args, log_path

## 3. 指标函数

In [ ]:
def as_cpu_float(x):
    if torch.is_tensor(x):
        return x.detach().cpu().float()
    return torch.as_tensor(x).detach().cpu().float()


def entropy_from_probs(probs, dim=-1):
    probs = probs.clamp_min(1.0e-12)
    return -(probs * probs.log()).sum(dim=dim)


def log_from_probs(probs):
    return probs.clamp_min(1.0e-12).log()


def accuracy_percent_from_logits(logits, labels):
    logits = as_cpu_float(logits)
    labels = torch.as_tensor(labels).cpu().long()
    pred = logits.argmax(dim=-1)
    return float((pred == labels).float().mean().item() * 100.0)


def nll_from_logits(logits, labels):
    logits = as_cpu_float(logits)
    labels = torch.as_tensor(labels).cpu().long()
    return float(F.cross_entropy(logits, labels, reduction="mean").item())


def brier_from_logits(logits, labels):
    logits = as_cpu_float(logits)
    labels = torch.as_tensor(labels).cpu().long()
    probs = F.softmax(logits, dim=-1)
    one_hot = F.one_hot(labels, num_classes=logits.shape[-1]).float()
    return float(((probs - one_hot) ** 2).sum(dim=-1).mean().item())


def calibration_bins_from_logits(logits, labels, n_bins=N_BINS):
    logits = as_cpu_float(logits)
    labels = torch.as_tensor(labels).cpu().long()
    probs = F.softmax(logits, dim=-1)
    conf, pred = probs.max(dim=-1)
    correct = (pred == labels).float()

    edges = torch.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    ece = 0.0
    total = int(labels.numel())

    for i in range(n_bins):
        left, right = edges[i], edges[i + 1]
        if i == 0:
            mask = (conf >= left) & (conf <= right)
        else:
            mask = (conf > left) & (conf <= right)

        count = int(mask.sum().item())
        frac = count / max(total, 1)

        if count > 0:
            avg_conf = float(conf[mask].mean().item() * 100.0)
            avg_acc = float(correct[mask].mean().item() * 100.0)
            correct_count = int(correct[mask].sum().item())
            gap = abs(avg_acc - avg_conf)
        else:
            avg_conf = 0.0
            avg_acc = 0.0
            correct_count = 0
            gap = 0.0

        weighted_gap = gap * frac
        ece += weighted_gap
        rows.append({
            "bin": i,
            "range_left": float(left.item()),
            "range_right": float(right.item()),
            "count": count,
            "fraction": frac,
            "correct_count": correct_count,
            "avg_confidence": avg_conf,
            "avg_accuracy": avg_acc,
            "gap": gap,
            "weighted_gap": weighted_gap,
        })

    return float(ece), rows


def confidence_entropy_margin_from_logits(logits):
    logits = as_cpu_float(logits)
    probs = F.softmax(logits, dim=-1)
    conf = probs.max(dim=-1).values
    ent = entropy_from_probs(probs, dim=-1)
    top2 = torch.topk(probs, k=min(2, probs.shape[-1]), dim=-1).values
    if top2.shape[-1] < 2:
        margin = torch.ones_like(conf)
    else:
        margin = top2[..., 0] - top2[..., 1]
    return {
        "confidence_mean": float(conf.mean().item() * 100.0),
        "entropy_mean": float(ent.mean().item()),
        "margin_mean": float(margin.mean().item() * 100.0),
    }


def metric_dict_from_logits(logits, labels):
    ece, _ = calibration_bins_from_logits(logits, labels, n_bins=N_BINS)
    return {
        "accuracy": accuracy_percent_from_logits(logits, labels),
        "ece": ece,
        "nll": nll_from_logits(logits, labels),
        "brier": brier_from_logits(logits, labels),
        **confidence_entropy_margin_from_logits(logits),
        "num_samples": int(torch.as_tensor(labels).numel()),
    }


def metric_row(method, split, mode, branch_or_fusion, logits, labels):
    return {
        "method": method,
        "split": split,
        "mode": mode,
        "branch_or_fusion": branch_or_fusion,
        **metric_dict_from_logits(logits, labels),
    }


def binary_auroc(scores, targets):
    scores = as_cpu_float(scores).reshape(-1)
    targets = torch.as_tensor(targets).cpu().long().reshape(-1)
    pos = targets == 1
    neg = targets == 0
    n_pos = int(pos.sum().item())
    n_neg = int(neg.sum().item())
    if n_pos == 0 or n_neg == 0:
        return float("nan")

    order = torch.argsort(scores, stable=True)
    sorted_scores = scores[order]
    ranks = torch.empty_like(scores, dtype=torch.float64)

    start = 0
    n = int(scores.numel())
    while start < n:
        end = start + 1
        while end < n and sorted_scores[end] == sorted_scores[start]:
            end += 1
        avg_rank = 0.5 * (float(start + 1) + float(end))
        ranks[order[start:end]] = avg_rank
        start = end

    sum_pos_ranks = ranks[pos].sum()
    auc = (sum_pos_ranks - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    return float(auc.item())


def aurc_eaurc_from_uncertainty(uncertainty, errors):
    uncertainty = as_cpu_float(uncertainty).reshape(-1)
    errors = torch.as_tensor(errors).cpu().float().reshape(-1)
    n = int(errors.numel())
    if n == 0:
        return float("nan"), float("nan")

    order = torch.argsort(uncertainty, descending=False, stable=True)
    sorted_errors = errors[order]
    counts = torch.arange(1, n + 1).float()
    risk = torch.cumsum(sorted_errors, dim=0) / counts
    aurc = float(risk.mean().item())

    optimal_errors = torch.sort(errors, descending=False).values
    optimal_risk = torch.cumsum(optimal_errors, dim=0) / counts
    eaurc = aurc - float(optimal_risk.mean().item())
    return aurc, eaurc


def fit_temperature_lbfgs(val_logits, val_labels, max_iter=50, device=None):
    val_logits = as_cpu_float(val_logits)
    val_labels = torch.as_tensor(val_labels).cpu().long()
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    logits = val_logits.to(device)
    labels = val_labels.to(device)
    log_t = torch.nn.Parameter(torch.zeros((), device=device))
    opt = torch.optim.LBFGS([log_t], max_iter=max_iter, line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        t = log_t.exp().clamp_min(1e-6)
        loss = F.cross_entropy(logits / t, labels)
        loss.backward()
        return loss

    opt.step(closure)
    t = float(log_t.exp().detach().cpu().item())
    if not math.isfinite(t) or t <= 0:
        return 1.0
    return t

## 4. 官方 `trainer.test()` sanity check

In [ ]:
def reset_eval_seed():
    random.seed(EVAL_SEED)
    np.random.seed(EVAL_SEED)
    torch.manual_seed(EVAL_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(EVAL_SEED)


def read_official_test_metrics(output_dir: Path):
    metrics_path = Path(output_dir) / "test_metrics.csv"
    if not metrics_path.exists():
        raise FileNotFoundError(f"trainer.test 没有生成 test_metrics.csv: {metrics_path}")
    df = pd.read_csv(metrics_path)
    if len(df) == 0:
        raise RuntimeError(f"test_metrics.csv 为空: {metrics_path}")
    row = df.iloc[0].to_dict()
    return row, metrics_path


def run_official_test_sanity(trainer, case_name: str):
    print(f"\n===== Official trainer.test sanity: {case_name} =====")
    reset_eval_seed()
    acc_ret = trainer.test(split="test")
    row, metrics_path = read_official_test_metrics(Path(trainer.cfg.OUTPUT_DIR))
    print("official metrics path:", metrics_path)
    print("trainer.test return acc:", acc_ret)
    print("official raw metrics:",
          {k: row.get(k) for k in ["accuracy", "ece", "nll", "brier", "num_samples"] if k in row})
    return row, metrics_path

## 5. 手工收集 logits / MC stack

In [ ]:
def cat0(chunks):
    if len(chunks) == 0:
        return None
    if len(chunks) == 1:
        return chunks[0]
    return torch.cat(chunks, dim=0)


def cat_mc(chunks):
    # chunks: list of [S, B, C] -> [S, N, C]
    if len(chunks) == 0:
        return None
    if len(chunks) == 1:
        return chunks[0]
    return torch.cat(chunks, dim=1)


def get_loader_for_split(trainer, split):
    if split == "val":
        if getattr(trainer, "val_loader", None) is None:
            return None
        return trainer.val_loader
    return trainer.test_loader


def build_eval_context(trainer, split):
    if hasattr(trainer, "executor") and hasattr(trainer.executor, "build_eval_context"):
        return trainer.executor.build_eval_context(trainer, split)
    raise RuntimeError("trainer.executor.build_eval_context 不存在。")


def select_eval_logits(trainer, outputs, eval_ctx):
    if hasattr(trainer.method, "select_eval_logits"):
        return trainer.method.select_eval_logits(outputs, eval_ctx)
    if hasattr(trainer.executor, "_select_eval_logits"):
        return trainer.executor._select_eval_logits(outputs, eval_ctx)
    return outputs.logits


def infer_num_mc_for_bayes_text(trainer):
    if N_MC_DIAG is not None:
        return int(N_MC_DIAG)
    method = trainer.method
    if hasattr(method, "n_mc_test"):
        return int(method.n_mc_test)
    cfg = trainer.cfg
    if hasattr(cfg, "BAYES_TEXT_MMRL"):
        return int(cfg.BAYES_TEXT_MMRL.N_MC_TEST)
    return 10


def _slice_classes(x, num_classes):
    if x is None:
        return None
    if x.dim() == 3:
        return x[..., :num_classes]
    return x[:, :num_classes]


@torch.no_grad()
def collect_official_case_outputs(trainer, case_name: str, split: str):
    # Important: each batch calls method.forward_eval exactly once.
    # Do not call extra Bayes diagnostic forwards here, otherwise RNG stream differs from trainer.test().
    loader = get_loader_for_split(trainer, split)
    if loader is None:
        print(f"[{case_name}] split={split}: val_loader 不存在，跳过。")
        return None

    trainer.set_model_mode("eval")
    eval_ctx = build_eval_context(trainer, split)
    method = trainer.method
    method_name = str(getattr(method, "method_name", getattr(trainer.cfg.METHOD, "NAME", "")))
    num_classes = int(getattr(method, "num_classes", 0))
    if num_classes <= 0:
        raise RuntimeError(f"[{case_name}] 无法获取 num_classes。")

    alpha = float(getattr(getattr(trainer.cfg, "BAYES_TEXT_MMRL", getattr(trainer.cfg, "MMRL", None)), "ALPHA", 0.7))

    print(f"[{case_name}] official collect split={split}, method={method_name}, num_classes={num_classes}")

    label_chunks = []
    repo_logits_chunks = []
    main_chunks = []
    rep_chunks = []
    fusion_chunks = []

    for batch_idx, batch in enumerate(loader):
        if MAX_BATCHES is not None and batch_idx >= int(MAX_BATCHES):
            break

        image = batch["img"].to(trainer.device)
        label = batch["label"].to(trainer.device)
        label_chunks.append(label.detach().cpu())

        outputs = method.forward_eval({"img": image, "label": label}, eval_ctx)
        routed = select_eval_logits(trainer, outputs, eval_ctx)

        repo_logits_chunks.append(_slice_classes(routed, num_classes).detach().cpu())
        main_chunks.append(_slice_classes(outputs.logits, num_classes).detach().cpu())
        rep_chunks.append(_slice_classes(outputs.aux_logits.get("rep"), num_classes).detach().cpu())
        fusion_chunks.append(_slice_classes(outputs.aux_logits.get("fusion"), num_classes).detach().cpu())

    labels = cat0(label_chunks)
    return {
        "case_name": case_name,
        "method_name": method_name,
        "split": split,
        "num_classes": num_classes,
        "alpha": alpha,
        "labels": labels,
        "repo_logits": cat0(repo_logits_chunks),
        "logits_main": cat0(main_chunks),
        "logits_rep": cat0(rep_chunks),
        "logits_fusion": cat0(fusion_chunks),
    }


@torch.no_grad()
def collect_bayes_text_diagnostics(trainer, case_name: str, split: str):
    # This is a separate second pass. It is not used for official sanity matching.
    loader = get_loader_for_split(trainer, split)
    if loader is None:
        return {}

    method = trainer.method
    method_name = str(getattr(method, "method_name", getattr(trainer.cfg.METHOD, "NAME", "")))
    is_bayes_text = method_name == "BayesTextMMRL" or hasattr(method.model, "forward_bayes_text")
    if not is_bayes_text:
        return {}

    trainer.set_model_mode("eval")
    num_classes = int(getattr(method, "num_classes", 0))
    n_mc = infer_num_mc_for_bayes_text(trainer)
    alpha = float(getattr(getattr(trainer.cfg, "BAYES_TEXT_MMRL", None), "ALPHA", 0.7))

    print(f"[{case_name}] diagnostic collect split={split}, n_mc={n_mc}")

    pm_main_chunks, pm_rep_chunks, pm_fusion_chunks = [], [], []
    mc_main_chunks, mc_rep_chunks, diag_current_fusion_chunks = [], [], []
    posterior_stats = {}
    text_mean_ref = None

    for batch_idx, batch in enumerate(loader):
        if MAX_BATCHES is not None and batch_idx >= int(MAX_BATCHES):
            break

        image = batch["img"].to(trainer.device)

        out_pm = method.model.forward_bayes_text(
            image,
            num_samples=1,
            use_posterior_mean=True,
            aggregation="prob_mean",
        )
        pm_main_chunks.append(_slice_classes(out_pm["logits"], num_classes).detach().cpu())
        pm_rep_chunks.append(_slice_classes(out_pm["logits_rep"], num_classes).detach().cpu())
        pm_fusion_chunks.append(_slice_classes(out_pm["logits_fusion"], num_classes).detach().cpu())

        out_mc = method.model.forward_bayes_text(
            image,
            num_samples=n_mc,
            use_posterior_mean=False,
            aggregation="prob_mean",
        )
        mc_main_stack = _slice_classes(out_mc["logits_stack"], num_classes)
        mc_rep_stack = _slice_classes(out_mc["logits_rep_stack"], num_classes)
        mc_main_chunks.append(mc_main_stack.detach().cpu())
        mc_rep_chunks.append(mc_rep_stack.detach().cpu())
        diag_current_fusion_chunks.append(_slice_classes(out_mc["logits_fusion"], num_classes).detach().cpu())

        if text_mean_ref is None:
            text_mean = out_mc["text_mean"][:num_classes].detach()
            prior_mean = method.text_features_clip[:num_classes].to(text_mean.device).detach()
            sigma = method.model.text_posterior.posterior_sigma().detach().to(text_mean.device)

            sigma_q2 = sigma.pow(2)
            sigma_p = method.model.text_posterior.prior_std[:num_classes].to(text_mean.device)
            sigma_p2 = sigma_p.pow(2)
            d = int(text_mean.shape[-1])
            mean_delta2 = (text_mean.float() - prior_mean.float()).pow(2).sum(dim=-1, keepdim=True)

            kl_per_class = 0.5 * (
                d * torch.log(sigma_p2 / sigma_q2)
                + (d * sigma_q2 + mean_delta2) / sigma_p2
                - d
            ).squeeze(-1)

            mean_shift = (text_mean.float() - prior_mean.float()).norm(dim=-1)
            snr = mean_shift / sigma.squeeze(-1).clamp_min(1e-12)

            posterior_stats = {
                "posterior_sigma": sigma.squeeze(-1).detach().cpu(),
                "posterior_mean_shift_per_class": mean_shift.detach().cpu(),
                "posterior_kl_per_class": kl_per_class.detach().cpu(),
                "posterior_snr_per_class": snr.detach().cpu(),
                "text_prior_std": sigma_p.squeeze(-1).detach().cpu(),
                "text_feature_dim": int(d),
                "alpha": alpha,
                "n_mc": n_mc,
            }
            text_mean_ref = text_mean.detach().cpu()

    diagnostics = {
        "pm_logits_main": cat0(pm_main_chunks),
        "pm_logits_rep": cat0(pm_rep_chunks),
        "pm_logits_fusion_current": cat0(pm_fusion_chunks),
        "mc_logits_main": cat_mc(mc_main_chunks),
        "mc_logits_rep": cat_mc(mc_rep_chunks),
        "diag_mc_logits_fusion_current": cat0(diag_current_fusion_chunks),
    }
    diagnostics.update(posterior_stats)
    return diagnostics


def save_raw_npz(case_name, split, data):
    if data is None or not SAVE_NPZ_CACHE:
        return None
    npz_data = {}
    for k, v in data.items():
        if torch.is_tensor(v):
            npz_data[k] = v.detach().cpu().numpy()
        elif isinstance(v, (int, float, str, bool)):
            npz_data[k] = np.array(v)
    path = OUT_DIR / f"{case_name}_{split}_raw_outputs.npz"
    np.savez_compressed(path, **npz_data)
    return path

## 6. 执行官方 sanity + 手工收集 + 强制对齐检查

In [ ]:
CASES = [
    {"name": "MMRL", "case_root": MMRL_CASE_ROOT},
    {"name": "BayesTextMMRL", "case_root": BAYES_CASE_ROOT},
]

raw_outputs = {}
official_metrics = {}
case_logs = {}
sanity_rows = []

for case in CASES:
    name = case["name"]
    trainer, args, parsed_args, log_path = build_and_load_trainer(name, case["case_root"])
    case_logs[name] = {
        "log_path": str(log_path),
        "parsed_args": parsed_args,
        "load_epoch": args.load_epoch,
        "load_epoch_source": args.load_epoch_source,
        "model_dir": args.model_dir,
    }

    # Step 1: 官方 evaluator sanity。
    official_row, official_metrics_path = run_official_test_sanity(trainer, name)
    official_metrics[name] = official_row

    # Step 2: official pass 手工收集。必须 reset seed，且每 batch 只调用一次 forward_eval。
    raw_outputs[name] = {}
    for split in SPLITS:
        reset_eval_seed()
        out = collect_official_case_outputs(trainer, name, split)
        if out is not None:
            raw_outputs[name][split] = out

    # Step 3: 比较 test official pass 与 trainer.test。
    test_out = raw_outputs[name].get("test")
    if test_out is None:
        raise RuntimeError(f"{name}: 没有收集到 test 输出。")

    manual_repo_metrics = metric_dict_from_logits(test_out["repo_logits"], test_out["labels"])

    off_acc = float(official_row["accuracy"])
    off_ece = float(official_row["ece"])
    off_nll = float(official_row["nll"])

    diff_acc = manual_repo_metrics["accuracy"] - off_acc
    diff_ece = manual_repo_metrics["ece"] - off_ece
    diff_nll = manual_repo_metrics["nll"] - off_nll

    proto = str(getattr(trainer.cfg.PROTOCOL, "NAME", ""))
    repo_fusion_max_abs_diff = float((test_out["repo_logits"] - test_out["logits_fusion"]).abs().max().item())
    repo_fusion_pred_agree = float((test_out["repo_logits"].argmax(1) == test_out["logits_fusion"].argmax(1)).float().mean().item() * 100.0)

    sanity_row = {
        "case_name": name,
        "method_name": test_out["method_name"],
        "protocol": proto,
        "split": "test",
        "official_accuracy": off_acc,
        "manual_repo_accuracy": manual_repo_metrics["accuracy"],
        "diff_accuracy_manual_minus_official": diff_acc,
        "official_ece": off_ece,
        "manual_repo_ece": manual_repo_metrics["ece"],
        "diff_ece_manual_minus_official": diff_ece,
        "official_nll": off_nll,
        "manual_repo_nll": manual_repo_metrics["nll"],
        "diff_nll_manual_minus_official": diff_nll,
        "repo_fusion_max_abs_diff": repo_fusion_max_abs_diff,
        "repo_fusion_pred_agree": repo_fusion_pred_agree,
        "official_metrics_path": str(official_metrics_path),
        "model_dir": args.model_dir,
        "load_epoch": args.load_epoch,
        "load_epoch_source": args.load_epoch_source,
    }
    sanity_rows.append(sanity_row)

    print("\nSanity row:")
    print(json.dumps({k: v for k, v in sanity_row.items() if k not in {"official_metrics_path", "model_dir"}}, indent=2, ensure_ascii=False))

    failed = (
        abs(diff_acc) > SANITY_TOL_ACC
        or abs(diff_ece) > SANITY_TOL_ECE
        or abs(diff_nll) > SANITY_TOL_NLL
    )

    if proto == "FS" and repo_fusion_max_abs_diff > 1e-5:
        failed = True

    # MMRL 是确定性模型，必须严格复现。
    # BayesTextMMRL 是 MC stochastic eval；两次 eval pass 的 raw metrics 可能不同，
    # 因此只记录差异，不作为 hard error。真正必须成立的是同一次 forward 内 FS repo==fusion。
    hard_fail = failed and (name == "MMRL")

    # 对 FS route invariant 单独处理：这个和 MC 随机性无关，所有方法都应成立。
    route_failed = (proto == "FS" and repo_fusion_max_abs_diff > 1e-5)

    if route_failed:
        raise RuntimeError(
            f"{name} route sanity failed: FS protocol expects repo_logits == fusion logits, "
            f"but max_abs_diff={repo_fusion_max_abs_diff}. See sanity row above."
        )

    if hard_fail and STRICT_OFFICIAL_SANITY:
        raise RuntimeError(
            f"{name} deterministic sanity check failed. "
            f"manual official-pass repo metrics do not match trainer.test. "
            f"See sanity row above."
        )

    if failed and name != "MMRL":
        print(
            f"[WARN] {name}: manual official-pass metrics differ from trainer.test. "
            "This is tolerated for MC stochastic eval; inspect official_sanity_check.csv."
        )

    # Step 4: sanity 通过后，第二遍单独收集 Bayes diagnostic。
    if name == "BayesTextMMRL":
        for split in list(raw_outputs[name].keys()):
            reset_eval_seed()
            diag = collect_bayes_text_diagnostics(trainer, name, split)
            raw_outputs[name][split].update(diag)

    # Step 5: 保存 raw outputs。
    for split, out in raw_outputs[name].items():
        npz_path = save_raw_npz(name, split, out)
        if npz_path:
            print(f"[saved] {npz_path}")

    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

sanity_df = pd.DataFrame(sanity_rows)
sanity_csv = OUT_DIR / "official_sanity_check.csv"
sanity_df.to_csv(sanity_csv, index=False)
display(sanity_df)
print("[saved]", sanity_csv)

## 7. 构造 BayesTextMMRL 多种推理口径

In [ ]:
def build_modes_for_mmrl(mmrl_out):
    if mmrl_out is None:
        return {}
    return {
        "MMRL_repo": mmrl_out["repo_logits"],
        "MMRL_main": mmrl_out["logits_main"],
        "MMRL_rep": mmrl_out["logits_rep"],
        "MMRL_fusion": mmrl_out["logits_fusion"],
    }


def majority_vote_logits_from_sample_preds(pred_s, num_classes):
    pred_s = torch.as_tensor(pred_s).cpu().long()
    S, N = pred_s.shape
    votes = torch.zeros(N, num_classes, dtype=torch.float32)
    for c in range(num_classes):
        votes[:, c] = (pred_s == c).float().sum(dim=0)
    probs = votes / float(max(S, 1))
    return log_from_probs(probs)


def build_modes_for_bayes(bt_out):
    if bt_out is None:
        return {}, {}

    alpha = float(bt_out.get("alpha", 0.7))
    num_classes = int(bt_out["num_classes"])

    mc_main = as_cpu_float(bt_out["mc_logits_main"])  # [S, N, C]
    mc_rep = as_cpu_float(bt_out["mc_logits_rep"])    # [S, N, C]
    p_main_s = F.softmax(mc_main, dim=-1)
    p_rep_s = F.softmax(mc_rep, dim=-1)

    p_main = p_main_s.mean(dim=0)
    p_rep = p_rep_s.mean(dim=0)
    z_main_logit_mean = mc_main.mean(dim=0)
    z_rep_logit_mean = mc_rep.mean(dim=0)

    p_arith_s = alpha * p_main_s + (1.0 - alpha) * p_rep_s
    p_arith = p_arith_s.mean(dim=0)

    geo_score = alpha * log_from_probs(p_main) + (1.0 - alpha) * log_from_probs(p_rep)
    z_geo = F.log_softmax(geo_score, dim=-1)
    z_logit_linear = alpha * z_main_logit_mean + (1.0 - alpha) * z_rep_logit_mean

    pred_fusion_s = p_arith_s.argmax(dim=-1)
    z_vote = majority_vote_logits_from_sample_preds(pred_fusion_s, num_classes)

    modes = {
        # official forward_eval routed output
        "BT_repo": bt_out["repo_logits"],
        "BT_forward_eval_main": bt_out["logits_main"],
        "BT_forward_eval_rep": bt_out["logits_rep"],
        "BT_forward_eval_fusion": bt_out["logits_fusion"],

        # same-stack diagnostic outputs
        "BT_diag_current_repo_style_fusion": bt_out["diag_mc_logits_fusion_current"],

        "BT_PM_main": bt_out["pm_logits_main"],
        "BT_PM_rep": bt_out["pm_logits_rep"],
        "BT_PM_fusion_current": bt_out["pm_logits_fusion_current"],

        "BT_MC_prob_main": log_from_probs(p_main),
        "BT_MC_prob_rep": log_from_probs(p_rep),
        "BT_arith_prob_fusion": log_from_probs(p_arith),

        "BT_MC_logit_main": z_main_logit_mean,
        "BT_MC_logit_rep": z_rep_logit_mean,
        "BT_logit_linear_fusion": z_logit_linear,

        "BT_geo_logprob_fusion": z_geo,
        "BT_majority_vote_fusion": z_vote,
    }

    sample_probs = {
        "BT_MC_prob_main": p_main_s,
        "BT_MC_prob_rep": p_rep_s,
        "BT_arith_prob_fusion": p_arith_s,
    }
    return modes, sample_probs


def build_all_modes(raw_outputs, split):
    modes = {}
    sample_probs = {}

    mmrl_out = raw_outputs.get("MMRL", {}).get(split)
    bt_out = raw_outputs.get("BayesTextMMRL", {}).get(split)

    if mmrl_out is not None:
        modes.update(build_modes_for_mmrl(mmrl_out))
        labels = mmrl_out["labels"]
    elif bt_out is not None:
        labels = bt_out["labels"]
    else:
        return {}, {}, None

    if bt_out is not None:
        bt_modes, bt_sample_probs = build_modes_for_bayes(bt_out)
        modes.update(bt_modes)
        sample_probs.update(bt_sample_probs)

    return modes, sample_probs, labels


all_modes = {}
all_sample_probs = {}
all_labels = {}

for split in SPLITS:
    modes, sample_probs, labels = build_all_modes(raw_outputs, split)
    if labels is not None:
        all_modes[split] = modes
        all_sample_probs[split] = sample_probs
        all_labels[split] = labels
        print(split, "modes:", list(modes.keys()))

## 8. 生成 8 张诊断表

In [ ]:
# 表 1: metrics_by_mode + calibration bins
metrics_rows = []
bins_rows = []

for split, modes in all_modes.items():
    labels = all_labels[split]
    for mode_name, logits in modes.items():
        method = "BayesTextMMRL" if mode_name.startswith("BT_") else "MMRL"
        metrics_rows.append(metric_row(method, split, mode_name, mode_name, logits, labels))

        ece, bins = calibration_bins_from_logits(logits, labels, n_bins=N_BINS)
        for b in bins:
            bins_rows.append({
                "method": method,
                "split": split,
                "mode": mode_name,
                **b,
            })

metrics_df = pd.DataFrame(metrics_rows).sort_values(["split", "method", "mode"])
bins_df = pd.DataFrame(bins_rows).sort_values(["split", "method", "mode", "bin"])

metrics_csv = OUT_DIR / "metrics_by_mode.csv"
bins_csv = OUT_DIR / "calibration_bins_by_mode.csv"
metrics_df.to_csv(metrics_csv, index=False)
bins_df.to_csv(bins_csv, index=False)

display(metrics_df)
print("[saved]", metrics_csv)
print("[saved]", bins_csv)

In [ ]:
# 表 2: posterior summary
posterior_rows = []
for split in SPLITS:
    bt = raw_outputs.get("BayesTextMMRL", {}).get(split)
    if bt is None:
        continue

    sigma = as_cpu_float(bt["posterior_sigma"]).reshape(-1)
    shift = as_cpu_float(bt["posterior_mean_shift_per_class"]).reshape(-1)
    kl = as_cpu_float(bt["posterior_kl_per_class"]).reshape(-1)
    snr = as_cpu_float(bt["posterior_snr_per_class"]).reshape(-1)
    d = int(bt.get("text_feature_dim", 1))
    c = int(sigma.numel())

    posterior_rows.append({
        "split": split,
        "num_classes": c,
        "text_feature_dim": d,
        "n_mc": int(bt.get("n_mc", 0)),
        "alpha": float(bt.get("alpha", 0.7)),
        "sigma_mean": float(sigma.mean().item()),
        "sigma_std": float(sigma.std(unbiased=False).item()),
        "sigma_min": float(sigma.min().item()),
        "sigma_max": float(sigma.max().item()),
        "sigma_cv": float((sigma.std(unbiased=False) / sigma.mean().clamp_min(1e-12)).item()),
        "mean_shift_mean": float(shift.mean().item()),
        "mean_shift_std": float(shift.std(unbiased=False).item()),
        "mean_shift_max": float(shift.max().item()),
        "KL_total": float(kl.sum().item()),
        "KL_per_cd": float(kl.sum().item() / max(c * d, 1)),
        "KL_class_mean": float(kl.mean().item()),
        "KL_class_std": float(kl.std(unbiased=False).item()),
        "SNR_mean": float(snr.mean().item()),
        "SNR_std": float(snr.std(unbiased=False).item()),
        "SNR_min": float(snr.min().item()),
        "SNR_max": float(snr.max().item()),
    })

posterior_df = pd.DataFrame(posterior_rows)
posterior_csv = OUT_DIR / "posterior_summary.csv"
posterior_df.to_csv(posterior_csv, index=False)
display(posterior_df)
print("[saved]", posterior_csv)

In [ ]:
def uncertainty_from_sample_probs(probs_s):
    probs_s = as_cpu_float(probs_s)
    S, N, C = probs_s.shape
    mean_p = probs_s.mean(dim=0)
    predictive_entropy = entropy_from_probs(mean_p, dim=-1)
    expected_entropy = entropy_from_probs(probs_s, dim=-1).mean(dim=0)
    mi = predictive_entropy - expected_entropy

    pred_s = probs_s.argmax(dim=-1)
    sample_agreement = torch.zeros(N)
    num_unique_preds = torch.zeros(N)
    for i in range(N):
        counts = torch.bincount(pred_s[:, i], minlength=C).float()
        sample_agreement[i] = counts.max() / float(S)
        num_unique_preds[i] = (counts > 0).float().sum()

    variation_ratio = 1.0 - sample_agreement
    conf_s = probs_s.max(dim=-1).values
    confidence_variance = conf_s.var(dim=0, unbiased=False)

    top2 = torch.topk(probs_s, k=min(2, C), dim=-1).values
    if top2.shape[-1] == 1:
        margin_s = torch.ones_like(conf_s)
    else:
        margin_s = top2[..., 0] - top2[..., 1]
    margin_variance = margin_s.var(dim=0, unbiased=False)

    return {
        "predictive_entropy": predictive_entropy,
        "expected_entropy": expected_entropy,
        "MI": mi,
        "sample_agreement": sample_agreement,
        "variation_ratio": variation_ratio,
        "confidence_variance": confidence_variance,
        "margin_variance": margin_variance,
        "num_unique_preds": num_unique_preds,
    }


def paired_decomposition(mmrl_logits, bayes_logits, labels, bayes_unc=None, comparison_name=""):
    labels = torch.as_tensor(labels).cpu().long()
    mmrl_logits = as_cpu_float(mmrl_logits)
    bayes_logits = as_cpu_float(bayes_logits)
    p_m = F.softmax(mmrl_logits, dim=-1)
    p_b = F.softmax(bayes_logits, dim=-1)

    pred_m = p_m.argmax(dim=-1)
    pred_b = p_b.argmax(dim=-1)
    corr_m = pred_m == labels
    corr_b = pred_b == labels
    conf_m = p_m.max(dim=-1).values
    conf_b = p_b.max(dim=-1).values

    gap_m = (corr_m.float() - conf_m).abs()
    gap_b = (corr_b.float() - conf_b).abs()

    masks = {
        "both_correct": corr_m & corr_b,
        "MMRL_correct_Bayes_wrong": corr_m & ~corr_b,
        "MMRL_wrong_Bayes_correct": ~corr_m & corr_b,
        "both_wrong": ~corr_m & ~corr_b,
    }

    rows = []
    for case_type, mask in masks.items():
        count = int(mask.sum().item())
        row = {
            "comparison": comparison_name,
            "case_type": case_type,
            "count": count,
            "fraction": float(count / max(int(labels.numel()), 1)),
        }
        if count > 0:
            row.update({
                "conf_mmrl": float(conf_m[mask].mean().item() * 100.0),
                "conf_bayes": float(conf_b[mask].mean().item() * 100.0),
                "conf_delta_bayes_minus_mmrl": float((conf_b[mask] - conf_m[mask]).mean().item() * 100.0),
                "sample_gap_mmrl": float(gap_m[mask].mean().item() * 100.0),
                "sample_gap_bayes": float(gap_b[mask].mean().item() * 100.0),
                "sample_gap_delta_bayes_minus_mmrl": float((gap_b[mask] - gap_m[mask]).mean().item() * 100.0),
            })
            if bayes_unc is not None:
                for k, v in bayes_unc.items():
                    vv = as_cpu_float(v).reshape(-1)
                    if vv.numel() == labels.numel():
                        row[f"bayes_{k}"] = float(vv[mask].mean().item())
        rows.append(row)
    return rows


# 表 3: paired decomposition
paired_rows = []
for split in SPLITS:
    if split not in all_modes:
        continue
    labels = all_labels[split]
    modes = all_modes[split]
    sample_probs = all_sample_probs.get(split, {})

    if "MMRL_repo" not in modes:
        continue

    bayes_unc = None
    if "BT_arith_prob_fusion" in sample_probs:
        bayes_unc = uncertainty_from_sample_probs(sample_probs["BT_arith_prob_fusion"])

    for bayes_mode in [
        "BT_repo",
        "BT_forward_eval_fusion",
        "BT_diag_current_repo_style_fusion",
        "BT_arith_prob_fusion",
        "BT_logit_linear_fusion",
        "BT_geo_logprob_fusion",
        "BT_PM_fusion_current",
        "BT_MC_prob_main",
    ]:
        if bayes_mode not in modes:
            continue
        paired_rows.extend(
            paired_decomposition(
                modes["MMRL_repo"],
                modes[bayes_mode],
                labels,
                bayes_unc=bayes_unc,
                comparison_name=f"{split}: MMRL_repo vs {bayes_mode}",
            )
        )

paired_df = pd.DataFrame(paired_rows)
paired_csv = OUT_DIR / "ece_paired_decomposition.csv"
paired_df.to_csv(paired_csv, index=False)
display(paired_df)
print("[saved]", paired_csv)

In [ ]:
# 表 4: uncertainty error detection
unc_rows = []
for split in SPLITS:
    if split not in all_modes:
        continue
    labels = all_labels[split]
    modes = all_modes[split]
    sample_probs = all_sample_probs.get(split, {})

    for mode_name, probs_s in sample_probs.items():
        if mode_name not in modes:
            continue
        logits = modes[mode_name]
        pred = as_cpu_float(logits).argmax(dim=-1)
        errors = (pred != labels).long()

        unc = uncertainty_from_sample_probs(probs_s)
        probs = F.softmax(as_cpu_float(logits), dim=-1)
        least_confidence = 1.0 - probs.max(dim=-1).values
        entropy = entropy_from_probs(probs, dim=-1)

        scores = {
            "least_confidence": least_confidence,
            "entropy": entropy,
            "predictive_entropy": unc["predictive_entropy"],
            "MI": unc["MI"],
            "variation_ratio": unc["variation_ratio"],
            "margin_variance": unc["margin_variance"],
            "confidence_variance": unc["confidence_variance"],
        }

        for score_name, score in scores.items():
            auroc = binary_auroc(score, errors)
            aurc, eaurc = aurc_eaurc_from_uncertainty(score, errors)
            unc_rows.append({
                "split": split,
                "mode": mode_name,
                "score": score_name,
                "AUROC_error": auroc,
                "AURC": aurc,
                "EAURC": eaurc,
                "num_errors": int(errors.sum().item()),
                "num_samples": int(errors.numel()),
                "score_mean_correct": float(score[errors == 0].mean().item()) if int((errors == 0).sum()) > 0 else float("nan"),
                "score_mean_wrong": float(score[errors == 1].mean().item()) if int((errors == 1).sum()) > 0 else float("nan"),
            })

uncertainty_df = pd.DataFrame(unc_rows)
uncertainty_csv = OUT_DIR / "uncertainty_error_detection.csv"
uncertainty_df.to_csv(uncertainty_csv, index=False)
display(uncertainty_df)
print("[saved]", uncertainty_csv)

In [ ]:
# 表 5/6: branch disagreement + fusion damage
def branch_disagreement_row(split, mode_prefix, z_main, z_rep, z_fusion, labels):
    labels = torch.as_tensor(labels).cpu().long()
    p_main = as_cpu_float(z_main).argmax(dim=-1)
    p_rep = as_cpu_float(z_rep).argmax(dim=-1)
    p_fusion = as_cpu_float(z_fusion).argmax(dim=-1)
    return {
        "split": split,
        "mode": mode_prefix,
        "main_rep_agree": float((p_main == p_rep).float().mean().item() * 100.0),
        "main_fusion_agree": float((p_main == p_fusion).float().mean().item() * 100.0),
        "rep_fusion_agree": float((p_rep == p_fusion).float().mean().item() * 100.0),
        "all_agree": float(((p_main == p_rep) & (p_main == p_fusion)).float().mean().item() * 100.0),
        "num_samples": int(labels.numel()),
    }


def fusion_damage_row(split, mode_prefix, z_main, z_rep, z_fusion, labels):
    labels = torch.as_tensor(labels).cpu().long()
    pred_main = as_cpu_float(z_main).argmax(dim=-1)
    pred_rep = as_cpu_float(z_rep).argmax(dim=-1)
    pred_fusion = as_cpu_float(z_fusion).argmax(dim=-1)
    c_main = pred_main == labels
    c_rep = pred_rep == labels
    c_fusion = pred_fusion == labels
    return {
        "split": split,
        "mode": mode_prefix,
        "fusion_correct_others_wrong": int((c_fusion & ~c_main & ~c_rep).sum().item()),
        "fusion_wrong_main_correct": int((~c_fusion & c_main).sum().item()),
        "fusion_wrong_rep_correct": int((~c_fusion & c_rep).sum().item()),
        "main_correct_rep_wrong_fusion_wrong": int((c_main & ~c_rep & ~c_fusion).sum().item()),
        "rep_correct_main_wrong_fusion_correct": int((c_rep & ~c_main & c_fusion).sum().item()),
        "fusion_correct_others_wrong_pct": float((c_fusion & ~c_main & ~c_rep).float().mean().item() * 100.0),
        "fusion_wrong_main_correct_pct": float((~c_fusion & c_main).float().mean().item() * 100.0),
        "fusion_wrong_rep_correct_pct": float((~c_fusion & c_rep).float().mean().item() * 100.0),
        "num_samples": int(labels.numel()),
    }


branch_rows = []
damage_rows = []

for split, modes in all_modes.items():
    labels = all_labels[split]
    branch_sets = []

    if all(k in modes for k in ["MMRL_main", "MMRL_rep", "MMRL_fusion"]):
        branch_sets.append(("MMRL", modes["MMRL_main"], modes["MMRL_rep"], modes["MMRL_fusion"]))

    if all(k in modes for k in ["BT_forward_eval_main", "BT_forward_eval_rep", "BT_forward_eval_fusion"]):
        branch_sets.append(("BT_forward_eval", modes["BT_forward_eval_main"], modes["BT_forward_eval_rep"], modes["BT_forward_eval_fusion"]))

    if all(k in modes for k in ["BT_PM_main", "BT_PM_rep", "BT_PM_fusion_current"]):
        branch_sets.append(("BT_PM_current", modes["BT_PM_main"], modes["BT_PM_rep"], modes["BT_PM_fusion_current"]))

    if all(k in modes for k in ["BT_MC_prob_main", "BT_MC_prob_rep", "BT_arith_prob_fusion"]):
        branch_sets.append(("BT_MC_arith_prob", modes["BT_MC_prob_main"], modes["BT_MC_prob_rep"], modes["BT_arith_prob_fusion"]))

    if all(k in modes for k in ["BT_MC_logit_main", "BT_MC_logit_rep", "BT_logit_linear_fusion"]):
        branch_sets.append(("BT_MC_logit_linear", modes["BT_MC_logit_main"], modes["BT_MC_logit_rep"], modes["BT_logit_linear_fusion"]))

    for name, z_main, z_rep, z_fusion in branch_sets:
        branch_rows.append(branch_disagreement_row(split, name, z_main, z_rep, z_fusion, labels))
        damage_rows.append(fusion_damage_row(split, name, z_main, z_rep, z_fusion, labels))

branch_df = pd.DataFrame(branch_rows)
damage_df = pd.DataFrame(damage_rows)

branch_csv = OUT_DIR / "branch_disagreement.csv"
damage_csv = OUT_DIR / "fusion_damage.csv"
branch_df.to_csv(branch_csv, index=False)
damage_df.to_csv(damage_csv, index=False)

display(branch_df)
display(damage_df)
print("[saved]", branch_csv)
print("[saved]", damage_csv)

In [ ]:
# 表 7: aggregation conflict
def aggregation_conflict_row(split, name_a, logits_a, name_b, logits_b, labels):
    labels = torch.as_tensor(labels).cpu().long()
    pred_a = as_cpu_float(logits_a).argmax(dim=-1)
    pred_b = as_cpu_float(logits_b).argmax(dim=-1)
    corr_a = pred_a == labels
    corr_b = pred_b == labels
    agree = pred_a == pred_b
    return {
        "split": split,
        "comparison": f"{name_a} vs {name_b}",
        "agree_pct": float(agree.float().mean().item() * 100.0),
        "A_correct_B_wrong": int((corr_a & ~corr_b).sum().item()),
        "A_wrong_B_correct": int((~corr_a & corr_b).sum().item()),
        "both_correct": int((corr_a & corr_b).sum().item()),
        "both_wrong": int((~corr_a & ~corr_b).sum().item()),
        "A_correct_B_wrong_pct": float((corr_a & ~corr_b).float().mean().item() * 100.0),
        "A_wrong_B_correct_pct": float((~corr_a & corr_b).float().mean().item() * 100.0),
        "num_samples": int(labels.numel()),
    }


conflict_pairs = [
    ("MMRL_repo", "MMRL_fusion"),
    ("BT_repo", "BT_forward_eval_fusion"),
    ("BT_PM_fusion_current", "BT_diag_current_repo_style_fusion"),
    ("BT_PM_fusion_current", "BT_arith_prob_fusion"),
    ("BT_diag_current_repo_style_fusion", "BT_arith_prob_fusion"),
    ("BT_arith_prob_fusion", "BT_geo_logprob_fusion"),
    ("BT_MC_prob_main", "BT_MC_logit_main"),
    ("BT_MC_prob_rep", "BT_MC_logit_rep"),
    ("BT_arith_prob_fusion", "BT_majority_vote_fusion"),
]

conflict_rows = []
for split, modes in all_modes.items():
    labels = all_labels[split]
    for a, b in conflict_pairs:
        if a in modes and b in modes:
            conflict_rows.append(aggregation_conflict_row(split, a, modes[a], b, modes[b], labels))

conflict_df = pd.DataFrame(conflict_rows)
conflict_csv = OUT_DIR / "aggregation_conflict.csv"
conflict_df.to_csv(conflict_csv, index=False)

display(conflict_df)
print("[saved]", conflict_csv)

In [ ]:
# 表 8: temperature scaling
temp_rows = []

if "val" in all_modes and "test" in all_modes:
    val_labels = all_labels["val"]
    test_labels = all_labels["test"]
    common_modes = sorted(set(all_modes["val"]).intersection(set(all_modes["test"])))

    for mode_name in common_modes:
        z_val = all_modes["val"][mode_name]
        z_test = all_modes["test"][mode_name]
        if z_val.shape[-1] != z_test.shape[-1]:
            continue

        T = fit_temperature_lbfgs(z_val, val_labels)
        z_test_scaled = as_cpu_float(z_test) / float(T)

        raw_row = metric_row(
            "BayesTextMMRL" if mode_name.startswith("BT_") else "MMRL",
            "test",
            mode_name,
            "raw",
            z_test,
            test_labels,
        )
        scaled_row = metric_row(
            "BayesTextMMRL" if mode_name.startswith("BT_") else "MMRL",
            "test",
            mode_name,
            "temperature_scaled",
            z_test_scaled,
            test_labels,
        )
        temp_rows.append({
            "mode": mode_name,
            "temperature": T,
            "raw_accuracy": raw_row["accuracy"],
            "raw_ece": raw_row["ece"],
            "raw_nll": raw_row["nll"],
            "raw_brier": raw_row["brier"],
            "scaled_accuracy": scaled_row["accuracy"],
            "scaled_ece": scaled_row["ece"],
            "scaled_nll": scaled_row["nll"],
            "scaled_brier": scaled_row["brier"],
            "ece_delta_scaled_minus_raw": scaled_row["ece"] - raw_row["ece"],
            "nll_delta_scaled_minus_raw": scaled_row["nll"] - raw_row["nll"],
        })
else:
    print("没有同时收集 val/test，跳过 temperature scaling。")

temp_df = pd.DataFrame(temp_rows)
temp_csv = OUT_DIR / "temperature_scaling_comparison.csv"
temp_df.to_csv(temp_csv, index=False)

display(temp_df)
print("[saved]", temp_csv)

## 9. 上传分析统表

In [ ]:
def infer_meta_from_case_root(case_root: Path):
    case_root = Path(case_root)
    parts = list(case_root.parts)
    meta = {
        "dataset": "",
        "shots": "",
        "backbone": "",
        "seed": "",
        "protocol": "",
        "phase": "",
    }
    try:
        if "output_refactor" in parts:
            i = parts.index("output_refactor")
            meta["protocol"] = parts[i + 2]
            meta["phase"] = parts[i + 3]
            meta["dataset"] = parts[i + 4]
            shots_token = parts[i + 5]
            meta["shots"] = int(shots_token.replace("shots_", "")) if shots_token.startswith("shots_") else shots_token
            meta["backbone"] = parts[i + 6]
            meta["seed"] = parts[i + 8] if len(parts) > i + 8 else ""
    except Exception:
        pass
    return meta


RUN_META = infer_meta_from_case_root(BAYES_CASE_ROOT)
ANALYSIS_RUN_ID = (
    f"{RUN_META.get('dataset','')}_shots{RUN_META.get('shots','')}_"
    f"{RUN_META.get('backbone','')}_{RUN_META.get('seed','')}_BayesTextMMRL_ECE_v4"
)

PRIMARY_MODES = [
    "MMRL_repo",
    "MMRL_fusion",
    "BT_repo",
    "BT_forward_eval_fusion",
    "BT_diag_current_repo_style_fusion",
    "BT_arith_prob_fusion",
    "BT_logit_linear_fusion",
    "BT_geo_logprob_fusion",
    "BT_PM_fusion_current",
    "BT_MC_prob_main",
    "BT_MC_prob_rep",
]

PRIMARY_METRICS = [
    "accuracy",
    "ece",
    "nll",
    "brier",
    "confidence_mean",
    "entropy_mean",
    "margin_mean",
]


def _first_float(df, filters, col, default=float("nan")):
    if df is None or len(df) == 0 or col not in df.columns:
        return default
    mask = pd.Series([True] * len(df))
    for k, v in filters.items():
        if k not in df.columns:
            return default
        mask &= (df[k] == v)
    sub = df.loc[mask]
    if len(sub) == 0:
        return default
    val = sub.iloc[0][col]
    try:
        return float(val)
    except Exception:
        return default


def build_summary_wide():
    rows = []
    available_splits = sorted(set(metrics_df["split"].unique())) if len(metrics_df) else []
    for split in available_splits:
        row = {
            "analysis_run_id": ANALYSIS_RUN_ID,
            **RUN_META,
            "split": split,
            "mmrl_case_root": str(MMRL_CASE_ROOT),
            "bayes_case_root": str(BAYES_CASE_ROOT),
        }

        for mode in PRIMARY_MODES:
            for metric in PRIMARY_METRICS:
                row[f"{mode}__{metric}"] = _first_float(metrics_df, {"split": split, "mode": mode}, metric)

        # Bayes 相对 official MMRL_repo 的 delta
        for mode in [m for m in PRIMARY_MODES if m.startswith("BT_")]:
            for metric in ["accuracy", "ece", "nll", "brier", "confidence_mean"]:
                base = row.get(f"MMRL_repo__{metric}", float("nan"))
                val = row.get(f"{mode}__{metric}", float("nan"))
                row[f"{mode}_minus_MMRL_repo__{metric}"] = val - base if pd.notna(base) and pd.notna(val) else float("nan")

        # official sanity
        for case_name in ["MMRL", "BayesTextMMRL"]:
            sub = sanity_df[sanity_df["case_name"] == case_name] if len(sanity_df) else pd.DataFrame()
            if len(sub):
                s = sub.iloc[0]
                for col in [
                    "official_accuracy", "manual_repo_accuracy", "diff_accuracy_manual_minus_official",
                    "official_ece", "manual_repo_ece", "diff_ece_manual_minus_official",
                    "official_nll", "manual_repo_nll", "diff_nll_manual_minus_official",
                    "repo_fusion_max_abs_diff", "repo_fusion_pred_agree",
                ]:
                    row[f"sanity__{case_name}__{col}"] = float(s[col])

        # posterior
        psub = posterior_df[posterior_df["split"] == split] if len(posterior_df) and "split" in posterior_df.columns else pd.DataFrame()
        if len(psub):
            p = psub.iloc[0]
            for col in posterior_df.columns:
                if col == "split":
                    continue
                if pd.api.types.is_numeric_dtype(posterior_df[col]):
                    row[f"posterior__{col}"] = float(p[col])

        # paired decomposition
        for bayes_mode in ["BT_repo", "BT_forward_eval_fusion", "BT_arith_prob_fusion", "BT_diag_current_repo_style_fusion"]:
            comparison_key = f"{split}: MMRL_repo vs {bayes_mode}"
            for case_type in ["both_correct", "MMRL_correct_Bayes_wrong", "MMRL_wrong_Bayes_correct", "both_wrong"]:
                filters = {"comparison": comparison_key, "case_type": case_type}
                prefix = f"paired__{bayes_mode}__{case_type}"
                for col in [
                    "count",
                    "fraction",
                    "conf_mmrl",
                    "conf_bayes",
                    "conf_delta_bayes_minus_mmrl",
                    "sample_gap_delta_bayes_minus_mmrl",
                    "bayes_MI",
                    "bayes_variation_ratio",
                    "bayes_sample_agreement",
                ]:
                    row[f"{prefix}__{col}"] = _first_float(paired_df, filters, col)

        # uncertainty
        for mode in ["BT_arith_prob_fusion", "BT_MC_prob_main", "BT_MC_prob_rep"]:
            for score in ["MI", "predictive_entropy", "entropy", "variation_ratio", "least_confidence", "margin_variance"]:
                filters = {"split": split, "mode": mode, "score": score}
                prefix = f"uncertainty__{mode}__{score}"
                for col in ["AUROC_error", "AURC", "EAURC", "score_mean_correct", "score_mean_wrong"]:
                    row[f"{prefix}__{col}"] = _first_float(uncertainty_df, filters, col)

        # branch/fusion
        for mode in ["MMRL", "BT_forward_eval", "BT_MC_arith_prob", "BT_PM_current"]:
            filters = {"split": split, "mode": mode}
            for col in ["main_rep_agree", "main_fusion_agree", "rep_fusion_agree", "all_agree"]:
                row[f"branch__{mode}__{col}"] = _first_float(branch_df, filters, col)
            for col in [
                "fusion_correct_others_wrong",
                "fusion_wrong_main_correct",
                "fusion_wrong_rep_correct",
                "fusion_wrong_main_correct_pct",
                "fusion_wrong_rep_correct_pct",
            ]:
                row[f"fusion_damage__{mode}__{col}"] = _first_float(damage_df, filters, col)

        # aggregation conflict
        for comparison in [
            "MMRL_repo vs MMRL_fusion",
            "BT_repo vs BT_forward_eval_fusion",
            "BT_diag_current_repo_style_fusion vs BT_arith_prob_fusion",
            "BT_PM_fusion_current vs BT_diag_current_repo_style_fusion",
            "BT_arith_prob_fusion vs BT_geo_logprob_fusion",
        ]:
            filters = {"split": split, "comparison": comparison}
            safe = comparison.replace(" ", "").replace("vs", "_vs_")
            for col in ["agree_pct", "A_correct_B_wrong", "A_wrong_B_correct", "A_correct_B_wrong_pct", "A_wrong_B_correct_pct"]:
                row[f"aggregation__{safe}__{col}"] = _first_float(conflict_df, filters, col)

        # temperature
        for mode in ["MMRL_repo", "BT_repo", "BT_forward_eval_fusion", "BT_arith_prob_fusion", "BT_diag_current_repo_style_fusion"]:
            filters = {"mode": mode}
            for col in ["temperature", "raw_ece", "scaled_ece", "ece_delta_scaled_minus_raw", "raw_nll", "scaled_nll", "nll_delta_scaled_minus_raw"]:
                row[f"temp__{mode}__{col}"] = _first_float(temp_df, filters, col)

        rows.append(row)
    return pd.DataFrame(rows)


summary_wide_df = build_summary_wide()
summary_wide_csv = OUT_DIR / "diagnosis_upload_summary_wide.csv"
summary_wide_df.to_csv(summary_wide_csv, index=False)

display(summary_wide_df)
print("[saved]", summary_wide_csv)


ID_COL_CANDIDATES = [
    "split", "method", "mode", "branch_or_fusion", "comparison", "case_type",
    "score", "bin", "range_left", "range_right", "case_name",
]

def make_long_table(df, source_table):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    df = df.copy()
    id_cols = [c for c in ID_COL_CANDIDATES if c in df.columns]
    numeric_cols = []
    for c in df.columns:
        if c in id_cols:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_cols.append(c)
        elif c not in id_cols:
            id_cols.append(c)

    rows = []
    for _, r in df.iterrows():
        base = {
            "analysis_run_id": ANALYSIS_RUN_ID,
            "source_table": source_table,
            **RUN_META,
        }
        for c in id_cols:
            base[c] = r.get(c, "")
        for c in numeric_cols:
            value = r.get(c)
            if pd.isna(value):
                continue
            rows.append({**base, "metric_name": c, "metric_value": float(value)})
    return pd.DataFrame(rows)


long_parts = [
    make_long_table(sanity_df, "official_sanity_check"),
    make_long_table(metrics_df, "metrics_by_mode"),
    make_long_table(bins_df, "calibration_bins_by_mode"),
    make_long_table(posterior_df, "posterior_summary"),
    make_long_table(paired_df, "ece_paired_decomposition"),
    make_long_table(uncertainty_df, "uncertainty_error_detection"),
    make_long_table(branch_df, "branch_disagreement"),
    make_long_table(damage_df, "fusion_damage"),
    make_long_table(conflict_df, "aggregation_conflict"),
    make_long_table(temp_df, "temperature_scaling_comparison"),
]

master_long_df = pd.concat([x for x in long_parts if x is not None and len(x) > 0], ignore_index=True)
master_long_csv = OUT_DIR / "diagnosis_upload_master_long.csv"
master_long_df.to_csv(master_long_csv, index=False)

print("[saved]", master_long_csv)
display(master_long_df.head())

## 10. 保存 Excel bundle

In [ ]:
xlsx_path = OUT_DIR / "bayes_text_mmrl_ece_diagnosis_report.xlsx"
upload_xlsx_path = OUT_DIR / "diagnosis_upload_bundle.xlsx"

tables = {
    "official_sanity": sanity_df,
    "summary_wide": summary_wide_df,
    "metrics_by_mode": metrics_df,
    "calibration_bins": bins_df,
    "posterior_summary": posterior_df,
    "paired_decomposition": paired_df,
    "uncertainty_error": uncertainty_df,
    "branch_disagreement": branch_df,
    "fusion_damage": damage_df,
    "aggregation_conflict": conflict_df,
    "temperature_scaling": temp_df,
}

try:
    with pd.ExcelWriter(xlsx_path) as writer:
        for name, df in tables.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)
    print("[saved]", xlsx_path)
except Exception as e:
    print("写 report Excel 失败；CSV 已保存。错误：", repr(e))

try:
    with pd.ExcelWriter(upload_xlsx_path) as writer:
        summary_wide_df.to_excel(writer, sheet_name="summary_wide", index=False)
        master_long_df.head(1_000_000).to_excel(writer, sheet_name="master_long", index=False)
        sanity_df.to_excel(writer, sheet_name="official_sanity", index=False)
        metrics_df.to_excel(writer, sheet_name="metrics_by_mode", index=False)
        posterior_df.to_excel(writer, sheet_name="posterior", index=False)
        paired_df.to_excel(writer, sheet_name="paired", index=False)
        uncertainty_df.to_excel(writer, sheet_name="uncertainty", index=False)
    print("[saved]", upload_xlsx_path)
except Exception as e:
    print("写 upload bundle Excel 失败；CSV 已保存。错误：", repr(e))

print("\n输出目录：", OUT_DIR)
print("\n建议上传分析：")
print("1.", summary_wide_csv)
print("2.", master_long_csv)
print("3.", OUT_DIR / "official_sanity_check.csv")

## 11. 读取结果的优先级

先看：

```text
official_sanity_check.csv
```

必须满足：

```text
MMRL official_accuracy ≈ manual_repo_accuracy
MMRL official_ece ≈ manual_repo_ece
MMRL repo_fusion_max_abs_diff ≈ 0
```

如果不满足，后面的 BayesTextMMRL 分析全部不可信。

然后看：

```text
diagnosis_upload_summary_wide.csv
```

重点比较：

```text
MMRL_repo__ece
BT_repo__ece
BT_arith_prob_fusion__ece
BT_forward_eval_fusion__ece
```

以及：

```text
posterior__sigma_mean
posterior__SNR_mean
uncertainty__BT_arith_prob_fusion__MI__AUROC_error
aggregation__BT_diag_current_repo_style_fusion_vs_BT_arith_prob_fusion__agree_pct
```